# Fig. 2 pnut(RNAi) DAPI/Fas3 manual label correction in napari

Cleaned from `RingCanals/Code/Pnut_napari.ipynb`. This notebook creates an initial DAPI nuclear segmentation for pnut(RNAi) follicle-cell ploidy measurements and supports manual correction in napari.

The final labels feed the Fig. 2I DAPI ploidy analysis.


## Imports

Figure association: upstream label generation for Fig. 2I.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import napari
import numpy as np
from csbdeep.io import save_tiff_imagej_compatible
from scipy import ndimage
from skimage import filters, morphology, segmentation
from skimage.morphology import ball, erosion, remove_small_objects
from tifffile import imread

from utils.RemoveRegion import remove_region
from utils.SplitLabels import SplitLabels

%matplotlib inline


## Paths and channel settings


In [ ]:
# Fig. 2I input image and output label path.
data_directory = Path("../Images/Pnut_phenotype/Experimental")
results_directory = Path("../Images/Pnut_phenotype/Results")
expt_name = "Pnut_experimental"

image_path = data_directory / "10930ts_pnut(RNAi)_fas3vasa_slide01032025_exp001p.tif"
label_save_path = results_directory / "control_same_exposure/10930ts_pnut(RNAi)_slide01032025_control002_labels.tif"

# The ImageJ export used by the source notebook stores channels on axis 1; move them to the last axis.
img = imread(image_path)
img = np.moveaxis(img, 1, -1)

FAS3_CHANNEL = 0
VASA_CHANNEL = 1
DAPI_CHANNEL = 2


## Build an initial DAPI label image


In [ ]:
# Fig. 2I preprocessing: mask germline/Vasa signal so the DAPI threshold focuses on follicle cells.
dapi = img[..., DAPI_CHANNEL]
fas3 = img[..., FAS3_CHANNEL]
vasa = img[..., VASA_CHANNEL]

vasa_otsu = filters.threshold_otsu(vasa)
vasa_mask = vasa > vasa_otsu
for z in range(vasa_mask.shape[0]):
    vasa_mask[z] = ndimage.binary_fill_holes(vasa_mask[z])

dapi_without_vasa = dapi * ~vasa_mask
dapi_otsu = filters.threshold_otsu(dapi_without_vasa)
fas3_otsu = filters.threshold_otsu(fas3)

# Fas3 is useful for QC and manual correction, even though labels start from DAPI.
dapi_mask = dapi_without_vasa > dapi_otsu - 100
fas3_mask = fas3 > fas3_otsu


In [ ]:
# Fig. 2I initial segmentation: separate nuclei conservatively, then expand back into the DAPI mask.
dapi_mask_eroded = erosion(dapi_mask, ball(1))
dapi_mask_eroded = remove_small_objects(dapi_mask_eroded, 200)
dapi_labels = morphology.label(dapi_mask_eroded)
dapi_labels = segmentation.expand_labels(dapi_labels, distance=1)
dapi_labels = remove_small_objects(dapi_labels, 4000)
dapi_labels = dapi_labels * dapi_mask

print(f"Initial label count: {int(dapi_labels.max())}")


## Quick QC preview


In [ ]:
# Fig. 2I QC: compare Fas3 and DAPI masks on a representative z-slice.
z_slice = min(92, dapi_mask.shape[0] - 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(fas3_mask[z_slice], cmap="gray")
axes[0].set_title("Fas3 mask")
axes[1].imshow(dapi_mask[z_slice], cmap="gray")
axes[1].set_title("DAPI mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()


## Manual correction in napari

Run this cell interactively, edit the `DAPI` labels layer, then continue. The optional cells below record common scripted edits from the source notebook.


In [ ]:
# Fig. 2I manual review: edit the DAPI labels layer in napari, then close the viewer.
viewer = napari.Viewer()
viewer.add_image(img, name="Image", channel_axis=-1)
viewer.add_labels(dapi_labels, name="DAPI")
viewer.add_labels(fas3_mask, name="Fas3_mask", visible=False)
napari.run()


## Optional scripted label edits


In [ ]:
# Fig. 2I optional cleanup from the source notebook. Edit these lists after visual inspection.
labels_to_split = [41, 5, 17, 3, 36, 20, 21, 6]
labels_to_delete = [110]
regions_to_remove = [101]

if labels_to_split:
    dapi_labels = SplitLabels(dapi_labels, labels_to_split, erosion_radius=2)

if labels_to_delete:
    dapi_labels[np.isin(dapi_labels, labels_to_delete)] = 0

for target_label in regions_to_remove:
    dapi_labels = remove_region(dapi_labels, target_label)

dapi_labels = morphology.remove_small_objects(dapi_labels, 1000)

if "viewer" in globals() and "DAPI" in viewer.layers:
    viewer.layers["DAPI"].data = dapi_labels


## Save corrected labels


In [ ]:
# Fig. 2I output: save the corrected DAPI labels for ploidy quantification.
if "viewer" in globals() and "DAPI" in viewer.layers:
    dapi_labels = viewer.layers["DAPI"].data

label_save_path.parent.mkdir(parents=True, exist_ok=True)
save_tiff_imagej_compatible(label_save_path, dapi_labels, axes="ZYX")
print(f"Saved {label_save_path}")
